In [1]:

import llm_config as _cfg
from langchain_core.rate_limiters import InMemoryRateLimiter as _Lim
_limiter = _Lim(requests_per_second=0.25, check_every_n_seconds=0.1, max_bucket_size=1)
_original = getattr(_cfg, '_original_create', _cfg.create_chat_model)
_cfg._original_create = _original
def _throttled(**kwargs):
    model = _original(**kwargs)
    model.rate_limiter = _limiter
    return model.with_retry(stop_after_attempt=6, wait_exponential_jitter=True)
_cfg.create_chat_model = _throttled


## Learning Objectives

By the end of this notebook, you should be able to:

- Build a small evaluation set that exposes a prompt's weak spots.
- Compare a baseline prompt against a stricter one on the same cases.
- Apply few-shot examples to improve consistency on harder cases.
- Reuse this prompt-testing loop in your own projects.

In [2]:
# Keep the setup cell focused so the notebook is easy to rerun from the top.
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

from llm_config import create_chat_model


def print_outputs(cases: list[dict[str, str]], outputs: list[str]) -> None:
    for case, output in zip(cases, outputs):
        print("=" * 100)
        print(f"Case: {case['label']}")
        print(f"What to watch: {case['watch_for']}")
        print(output)

In [3]:
load_dotenv()

llm = create_chat_model(temperature=0.1, max_tokens=512)

output_parser = StrOutputParser()

## Step 1 - Create a Tiny Evaluation Set

For prompt testing, a tiny but varied test set is often more useful than one perfect example. The goal is to include a few cases that are different enough to expose prompt weaknesses.

In this notebook, we will use a simple support-triage task. Each case is a short customer message, and we want the model to:
- estimate urgency
- identify the topic
- suggest a short response


In [4]:
# Keep the test set small enough to inspect manually.
support_cases = [
    {
        "label": "Damaged package",
        "message": "My package arrived today and the mug inside was broken. I need a replacement before Friday because it is a gift.",
        "watch_for": "Should likely be high urgency and mention replacement or support steps.",
    },
    {
        "label": "Password reset",
        "message": "I cannot sign in to my account because I forgot my password. How do I reset it?",
        "watch_for": "Should be account-related and not overstate urgency.",
    },
    {
        "label": "Double charge",
        "message": "I think my card was charged twice for the same order. Can someone check this?",
        "watch_for": "Should identify billing and treat it as fairly urgent.",
    },
    {
        "label": "Sustainability question",
        "message": "Your website says your products are sustainable. What materials do you actually use?",
        "watch_for": "Should stay low urgency and answer as a product question, not a complaint.",
    },
]

support_cases

[{'label': 'Damaged package',
  'message': 'My package arrived today and the mug inside was broken. I need a replacement before Friday because it is a gift.',
  'watch_for': 'Should likely be high urgency and mention replacement or support steps.'},
 {'label': 'Password reset',
  'message': 'I cannot sign in to my account because I forgot my password. How do I reset it?',
  'watch_for': 'Should be account-related and not overstate urgency.'},
 {'label': 'Double charge',
  'message': 'I think my card was charged twice for the same order. Can someone check this?',
  'watch_for': 'Should identify billing and treat it as fairly urgent.'},
 {'label': 'Sustainability question',
  'message': 'Your website says your products are sustainable. What materials do you actually use?',
  'watch_for': 'Should stay low urgency and answer as a product question, not a complaint.'}]

A compact evaluation set like this helps you ask practical questions:
- Which cases produce messy or inconsistent output?
- Which cases are misclassified?
- Which instructions are too vague?

Notice that the test set mixes logistics, account access, billing, and product questions. That variety makes the prompt easier to stress-test.

## Step 2 - Start With a Baseline Prompt

We begin with a deliberately simple prompt. It is good enough to run, but it leaves a lot of room for the model to choose its own output structure.

In [5]:
# This baseline prompt defines the task but does not strongly constrain the format.
baseline_template = """You are helping an ecommerce support team.
Read the customer message and identify the urgency, the topic, and a short reply.

Customer message:
{message}
"""

baseline_prompt = PromptTemplate(
    template=baseline_template,
    input_variables=["message"],
)

baseline_chain = baseline_prompt | llm | output_parser

In [6]:
# Try the baseline prompt on one case before scaling up.
print(baseline_chain.invoke({"message": support_cases[0]["message"]}))

**Urgency:** High  
**Topic:** Damaged item – request for a replacement mug  
**Reply:**  
"I'm sorry the mug arrived broken. We'll ship a replacement immediately and arrange expedited delivery so it arrives before Friday. Let me know if there's anything else I can help with."


## Step 3 - Evaluate the Baseline Across All Cases

One output is not enough to judge a prompt. Prompt testing becomes more informative when you run the same prompt on several cases and compare the outputs side by side.

In [7]:
# batch() lets us reuse the same prompt on many cases at once.
baseline_inputs = [{"message": case["message"]} for case in support_cases]
baseline_outputs = baseline_chain.batch(baseline_inputs)
print_outputs(support_cases, baseline_outputs)

Case: Damaged package
What to watch: Should likely be high urgency and mention replacement or support steps.
**Urgency:** High – needs a replacement before Friday.  
**Topic:** Damaged item – request for a replacement mug.  

**Reply:**  
Hi [Customer Name],  

I’m really sorry to hear that the mug arrived broken. We’ll ship a replacement to you right away and will use expedited shipping so it arrives before Friday. I’ll send you a tracking number as soon as it’s on its way.  

Thank you for your patience, and please let me know if there’s anything else I can do for you.  

Best,  
[Your Name]  
E‑commerce Support Team
Case: Password reset
What to watch: Should be account-related and not overstate urgency.
**Urgency:** Medium  
**Topic:** Password reset / account access  

**Reply:**  
Hi there! No worries—just click the “Forgot password?” link on the login page, enter the email address linked to your account, and we’ll send you a reset link. Follow the instructions in the email to set

### What Usually Goes Wrong in the Baseline

With a loose prompt, common issues include:
- inconsistent formatting from one case to another
- uneven detail across cases
- vague urgency labels
- responses that are friendly but hard to compare systematically

This is a good sign that we need a stronger prompt specification, not necessarily a different model.

## Step 4 - Improve the Prompt With Stronger Instructions

Now we make the task more explicit. We still use the same model, but we add tighter rules for the output format so the results become easier to inspect and compare.

In [8]:
# Stronger instructions often improve consistency more than changing the model does.
improved_template = """You are triaging ecommerce support messages.
Analyze the customer message and return exactly this format:

Urgency: <low|medium|high>
Topic: <shipping|returns|billing|account|product>
Suggested reply:
- sentence 1
- sentence 2

Keep the reply concise and practical.

Customer message:
{message}
"""

improved_prompt = PromptTemplate(
    template=improved_template,
    input_variables=["message"],
)

improved_chain = improved_prompt | llm | output_parser

In [9]:
# Compare the improved prompt on the same evaluation set.
improved_outputs = improved_chain.batch(baseline_inputs)
print_outputs(support_cases, improved_outputs)

Case: Damaged package
What to watch: Should likely be high urgency and mention replacement or support steps.
Urgency: high  
Topic: product  
Suggested reply:  
- I'm sorry the mug arrived broken; we'll ship a replacement to you immediately.  
- We'll use expedited shipping so it arrives before Friday and I'll send you the tracking details.
Case: Password reset
What to watch: Should be account-related and not overstate urgency.
Urgency: low  
Topic: account  
Suggested reply:  
- You can reset your password by clicking the “Forgot password?” link on the login page and following the instructions sent to your email.  
- If you don’t receive the email within a few minutes, check your spam folder or let us know so we can assist further.
Case: Double charge
What to watch: Should identify billing and treat it as fairly urgent.
Urgency: medium  
Topic: billing  
Suggested reply:  
- I'm sorry to hear about the duplicate charge; could you please provide your order number so we can investigate?

#### Compare Two Prompt Variants on One Hard Case

Comparing full test sets is useful, but sometimes you also want to zoom in on one case that feels tricky and inspect how each prompt handles it.

In [10]:
hard_case = support_cases[2]["message"]

print("BASELINE")
print(baseline_chain.invoke({"message": hard_case}))
print("\n" + "-" * 100 + "\n")
print("IMPROVED")
print(improved_chain.invoke({"message": hard_case}))

BASELINE


**Urgency:** High  
**Topic:** Duplicate charge / billing issue  

**Reply:**  
"I'm sorry to hear that. Could you please share your order number so we can investigate the duplicate charge and issue a refund if needed?"

----------------------------------------------------------------------------------------------------

IMPROVED


Urgency: medium  
Topic: billing  
Suggested reply:  
- I’m sorry to hear about the duplicate charge; I’ll review your order right away.  
- I’ll update you within the next 24 hours and ensure the extra amount is refunded if needed.


## Step 5 - Add Few-Shot Examples for Harder Edge Cases

If the output is still inconsistent, few-shot examples can teach the model the exact style you want. This is especially useful when the format matters or when the task includes edge cases that the model may interpret differently.

```mermaid
flowchart LR
    A[Example input-output pairs] --> B[FewShotPromptTemplate]
    C[New customer message] --> B
    B --> D[Formatted prompt]
    D --> E[More controlled output]
```

The examples do not retrain the model. They just give the model a clearer pattern to imitate for the current request.

In [11]:
# Each example demonstrates the output style we want on a realistic support case.
examples = [
    {
        "message": "My order still has not arrived and I need it tomorrow for an event.",
        "analysis": "Urgency: high\nTopic: shipping\nSuggested reply:\n- I am sorry your order has not arrived in time for your event.\n- Please send us your order number so we can check shipping options right away.",
    },
    {
        "message": "I forgot my password and cannot get into my account.",
        "analysis": "Urgency: medium\nTopic: account\nSuggested reply:\n- Please use the password reset link on the sign-in page to create a new password.\n- If that does not work, contact support and we can help verify your account.",
    },
]

example_prompt = PromptTemplate(
    input_variables=["message", "analysis"],
    template="""Example message:\n{message}\n\nExample output:\n{analysis}""",
)

prefix = """You are triaging ecommerce support messages.
Return exactly this format:
Urgency: <low|medium|high>
Topic: <shipping|returns|billing|account|product>
Suggested reply:
- sentence 1
- sentence 2

Use the examples below as style guidance."""

suffix = """Now analyze this customer message:
{message}"""

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["message"],
    example_separator="\n\n",
)

few_shot_chain = few_shot_prompt | llm | output_parser

#### Inspect the Final Few-Shot Prompt

When working with more complex prompt templates, printing the final prompt is a great debugging habit. It lets you check whether the examples and the new input were assembled the way you expected.

In [12]:
print(few_shot_prompt.format(message=support_cases[0]["message"]))

You are triaging ecommerce support messages.
Return exactly this format:
Urgency: <low|medium|high>
Topic: <shipping|returns|billing|account|product>
Suggested reply:
- sentence 1
- sentence 2

Use the examples below as style guidance.

Example message:
My order still has not arrived and I need it tomorrow for an event.

Example output:
Urgency: high
Topic: shipping
Suggested reply:
- I am sorry your order has not arrived in time for your event.
- Please send us your order number so we can check shipping options right away.

Example message:
I forgot my password and cannot get into my account.

Example output:
Urgency: medium
Topic: account
Suggested reply:
- Please use the password reset link on the sign-in page to create a new password.
- If that does not work, contact support and we can help verify your account.

Now analyze this customer message:
My package arrived today and the mug inside was broken. I need a replacement before Friday because it is a gift.


In [13]:
# Evaluate the few-shot prompt on the same test set.
few_shot_outputs = few_shot_chain.batch(baseline_inputs)
print_outputs(support_cases, few_shot_outputs)

Case: Damaged package
What to watch: Should likely be high urgency and mention replacement or support steps.
Urgency: high  
Topic: product  
Suggested reply:  
- I’m sorry to hear the mug was broken when it arrived.  
- Please provide your order number and we’ll ship a replacement right away to ensure it arrives before Friday.
Case: Password reset
What to watch: Should be account-related and not overstate urgency.
Urgency: medium
Topic: account
Suggested reply:
- Please use the password reset link on the sign-in page to create a new password.
- If that does not work, contact support and we can help verify your account.
Case: Double charge
What to watch: Should identify billing and treat it as fairly urgent.
Urgency: medium  
Topic: billing  
Suggested reply:  
- I’m sorry to hear you’ve been charged twice for the same order.  
- Please provide your order number and the date of the transaction so we can investigate and resolve this promptly.
Case: Sustainability question
What to watch:

## Step 6 - Reuse the Prompt on a New Test Set

A prompt is more convincing when it generalizes beyond the first set of examples. Here we apply the same few-shot prompt to a second mini test set with different customer messages.

In [14]:
new_cases = [
    {
        "label": "Gift card request",
        "message": "Do you sell gift cards? I want to send one to a friend for their birthday.",
        "watch_for": "Should be product-related and low urgency.",
    },
    {
        "label": "Express shipping",
        "message": "I placed my order an hour ago. Can I upgrade it to express shipping?",
        "watch_for": "Should likely be shipping-related and time-sensitive.",
    },
]

new_outputs = few_shot_chain.batch([{"message": case["message"]} for case in new_cases])
print_outputs(new_cases, new_outputs)

Case: Gift card request
What to watch: Should be product-related and low urgency.
Urgency: low
Topic: product
Suggested reply:
- Yes, we offer digital gift cards that can be sent directly to your friend’s email.
- Let me know the amount you’d like to gift, and I’ll walk you through the checkout process.
Case: Express shipping
What to watch: Should likely be shipping-related and time-sensitive.
Urgency: medium  
Topic: shipping  
Suggested reply:  
- Thank you for reaching out; we can help you upgrade to express shipping.  
- Please provide your order number and we’ll check if the upgrade is available and apply it right away.


## Exercises

1. Add one more support case to `support_cases` and test all three prompt variants.
2. Change the improved prompt so it must return JSON-like output instead of bullet points.
3. Add one more few-shot example for a returns-related message and compare the result.
4. Raise the model temperature and check whether output consistency gets better or worse.


## My Exercise Answers

The cells below answer the four exercises in order. Each one reuses the chains and test cases defined above, so run the notebook from the top first.

### Exercise 1 - One More Support Case, All Three Prompt Variants

The new case is an address change on an order that ships tomorrow. It is a useful stress test because the customer is polite and calm, but the request is time-critical: a prompt that judges urgency by tone instead of by deadline will rate it too low.

In [15]:
# Add the new case once, even if this cell is run again.
address_case = {
    "label": "Address change",
    "message": "I just noticed my order is going to my old address. It ships tomorrow morning. Can you change the delivery address before it leaves?",
    "watch_for": "Should be shipping and high urgency: calm tone, but a hard deadline.",
}

if address_case not in support_cases:
    support_cases.append(address_case)

# Rebuild the inputs so every variant sees the same five cases.
all_inputs = [{"message": case["message"]} for case in support_cases]

variants = {
    "BASELINE": baseline_chain,
    "IMPROVED": improved_chain,
    "FEW-SHOT": few_shot_chain,
}

variant_outputs = {}
for name, chain in variants.items():
    variant_outputs[name] = chain.batch(all_inputs, config={"max_concurrency": 2})
    print("\n" + "#" * 40, name, "#" * 40)
    print_outputs(support_cases, variant_outputs[name])


######################################## BASELINE ########################################
Case: Damaged package
What to watch: Should likely be high urgency and mention replacement or support steps.
**Urgency:** High  
**Topic:** Damaged mug – request for a replacement before Friday  

**Reply:**  
Hi [Customer Name],  
I’m sorry to hear the mug arrived broken. We’ll ship a replacement right away and arrange expedited delivery so it arrives before Friday. I’ll send you the tracking details shortly. Thank you for your patience!
Case: Password reset
What to watch: Should be account-related and not overstate urgency.
**Urgency:** Low  
**Topic:** Password reset / account login  

**Reply:**  
Hi! No worries—just click the “Forgot password?” link on the login page, enter your email address, and we’ll send you a reset link. Once you click that link, you can set a new password. Let me know if you need any further help!
Case: Double charge
What to watch: Should identify billing and treat it


######################################## IMPROVED ########################################
Case: Damaged package
What to watch: Should likely be high urgency and mention replacement or support steps.
Urgency: high  
Topic: product  
Suggested reply:  
- I'm sorry the mug arrived broken; we'll ship a replacement to you immediately.  
- It should arrive by Thursday so you can give it as a gift on Friday.
Case: Password reset
What to watch: Should be account-related and not overstate urgency.
Urgency: low  
Topic: account  
Suggested reply:  
- Click the “Forgot password” link on the login page and follow the instructions sent to your email.  
- If you don’t receive the email, check your spam folder or contact support for further assistance.
Case: Double charge
What to watch: Should identify billing and treat it as fairly urgent.
Urgency: high  
Topic: billing  
Suggested reply:  
- I'm sorry to hear about the duplicate charge; let me look into this for you right away.  
- Could you plea


######################################## FEW-SHOT ########################################
Case: Damaged package
What to watch: Should likely be high urgency and mention replacement or support steps.
Urgency: high
Topic: product
Suggested reply:
- I'm sorry to hear the mug arrived broken and understand you need a replacement quickly.
- Please provide your order number and we will ship a new mug to you before Friday.
Case: Password reset
What to watch: Should be account-related and not overstate urgency.
Urgency: medium
Topic: account
Suggested reply:
- Please use the password reset link on the sign-in page to create a new password.
- If that does not work, contact support and we can help verify your account.
Case: Double charge
What to watch: Should identify billing and treat it as fairly urgent.
Urgency: medium  
Topic: billing  
Suggested reply:  
- I’m sorry to hear about the duplicate charge and will look into it right away.  
- Please provide your order number or the transaction 

Reading three long printouts side by side is slow, so the next cell pulls the `Urgency:` and `Topic:` lines out of each output and puts them in one small table. The baseline prompt has no fixed format, so its labels often cannot be extracted at all - which is itself the finding.

In [16]:
import re


def extract_label(output: str, field: str) -> str:
    """Return the value after 'Urgency:' or 'Topic:', or '?' if the format is missing."""
    match = re.search(rf"^\W*{field}\W*:\W*(\w+)", output, flags=re.IGNORECASE | re.MULTILINE)
    return match.group(1).lower() if match else "?"


print(f"{'Case':<26}" + "".join(f"{name:<22}" for name in variants))
for i, case in enumerate(support_cases):
    row = f"{case['label']:<26}"
    for name in variants:
        output = variant_outputs[name][i]
        row += f"{extract_label(output, 'Urgency') + ' / ' + extract_label(output, 'Topic'):<22}"
    print(row)

Case                      BASELINE              IMPROVED              FEW-SHOT              
Damaged package           high / damaged        high / product        high / product        
Password reset            low / password        low / account         medium / account      
Double charge             high / duplicate      high / billing        medium / billing      
Sustainability question   low / sustainability  low / product         low / product         
Address change            high / change         high / shipping       high / shipping       


**What I observed** (run of 2026-11-13, `openai/gpt-oss-20b`; a rerun will differ in detail)

- **The new case was classified correctly by both structured prompts:** `high / shipping` for the address change, despite the calm tone. The baseline also said high urgency.
- **The baseline has no shared topic vocabulary.** Its topics were free text ("Damaged mug – request for a replacement before Friday", "Duplicate charge / billing issue"), so the table can only show the first word. The answers are not wrong, they just cannot be compared or counted.
- **Few-shot changed labels, not only the format.** Password reset went from `low` to `medium` and double charge from `high` to `medium`. The password example in the few-shot prompt is labelled `medium`, and the model copied that label along with the layout. Examples teach content too, so their labels have to be chosen as carefully as their format.
- **"Damaged package" came out as `product` in both structured prompts.** None of the five allowed topics fits a damaged delivery cleanly; the test set exposed a gap in the label list, not in the model.
- **The replies invent facts.** A hotline number (1-800-123-4567), a delivery promise ("by Thursday") and a list of materials appear although the prompt contains none of this. Format rules do not prevent that; the prompt would need real context or an instruction to promise nothing it cannot know.

### Exercise 2 - JSON Output Instead of Bullet Points

The improved prompt now asks for one JSON object. Two details matter:

- Literal braces in a `PromptTemplate` must be doubled (`{{` and `}}`), otherwise LangChain reads them as input variables. `{message}` stays the only real variable.
- `JsonOutputParser` replaces `StrOutputParser`, so the chain returns a Python dictionary instead of text. That turns "the format looks right" into something code can check.

In [17]:
from langchain_core.output_parsers import JsonOutputParser

json_template = """You are triaging ecommerce support messages.
Analyze the customer message and return only one JSON object, with no text before or after it.
Use exactly these keys and allowed values:

{{
  "urgency": "low" | "medium" | "high",
  "topic": "shipping" | "returns" | "billing" | "account" | "product",
  "suggested_reply": ["sentence 1", "sentence 2"]
}}

Keep the reply concise and practical.

Customer message:
{message}
"""

json_prompt = PromptTemplate(
    template=json_template,
    input_variables=["message"],
)

json_chain = json_prompt | llm | JsonOutputParser()

json_outputs = json_chain.batch(all_inputs, config={"max_concurrency": 2})

for case, result in zip(support_cases, json_outputs):
    print("=" * 100)
    print(f"Case: {case['label']}")
    print(result)

Case: Damaged package
{'urgency': 'high', 'topic': 'returns', 'suggested_reply': ["We're sorry to hear the mug was damaged during shipping.", "We'll ship a replacement right away and confirm the delivery date before Friday."]}
Case: Password reset
{'urgency': 'medium', 'topic': 'account', 'suggested_reply': ["You can reset your password by clicking the 'Forgot password' link on the login page.", 'Follow the link sent to your email to set a new password.']}
Case: Double charge
{'urgency': 'high', 'topic': 'billing', 'suggested_reply': ['I’m sorry to hear about the duplicate charge.', 'I’ve opened a ticket and will review your transaction details and update you shortly.']}
Case: Sustainability question
{'urgency': 'medium', 'topic': 'product', 'suggested_reply': ['Thank you for reaching out.', 'We use a blend of recycled polyester, organic cotton, and responsibly sourced bamboo in our products; detailed material information is available on our sustainability page here: https://example.co

Because the output is now data, the format rules can be tested automatically instead of by eye:

In [18]:
ALLOWED_URGENCY = {"low", "medium", "high"}
ALLOWED_TOPIC = {"shipping", "returns", "billing", "account", "product"}


def check_triage(result: dict) -> list[str]:
    """Return a list of format problems; an empty list means the output is valid."""
    problems = []
    if set(result) != {"urgency", "topic", "suggested_reply"}:
        problems.append(f"unexpected keys: {sorted(result)}")
    if result.get("urgency") not in ALLOWED_URGENCY:
        problems.append(f"urgency not allowed: {result.get('urgency')!r}")
    if result.get("topic") not in ALLOWED_TOPIC:
        problems.append(f"topic not allowed: {result.get('topic')!r}")
    reply = result.get("suggested_reply")
    if not isinstance(reply, list) or len(reply) != 2:
        problems.append("suggested_reply is not a list of 2 sentences")
    return problems


for case, result in zip(support_cases, json_outputs):
    problems = check_triage(result)
    status = "OK" if not problems else "PROBLEM: " + "; ".join(problems)
    print(f"{case['label']:<26}{result.get('urgency', '?'):<8}{result.get('topic', '?'):<10}{status}")

Damaged package           high    returns   OK
Password reset            medium  account   OK
Double charge             high    billing   OK
Sustainability question   medium  product   OK
Address change            high    shipping  PROBLEM: suggested_reply is not a list of 2 sentences


**What I observed** (run of 2026-11-13, `openai/gpt-oss-20b`; a rerun will differ in detail)

- **All five outputs parsed as JSON, and all labels were inside the allowed values.** The doubled braces and `JsonOutputParser` work as intended.
- **The automatic check found one problem a quick read would miss:** for the address change, `suggested_reply` held one string with two sentences instead of a list of two. This is exactly what the check is for.
- **Changing only the output format changed the labels.** Compared with the bullet-point prompt, damaged package moved from `product` to `returns`, and password reset and the sustainability question moved from `low` to `medium`. A format change is a prompt change and needs the same test run as any other.
- **One reply claims an action that never happened** ("I've updated your shipping address ... will be delivered tomorrow"), and one invents a link (example.com). Valid JSON says nothing about whether the content is true.

### Exercise 3 - A Returns Example for the Few-Shot Prompt

The two original examples cover `shipping` and `account` only. A third example now shows a returns message. To compare fairly, the old and the new few-shot prompt run on the same three returns-flavoured messages, including two where returns competes with another topic (a damaged delivery, a refund that has not been paid).

In [19]:
returns_example = {
    "message": "The jacket I ordered is too small. How can I send it back?",
    "analysis": "Urgency: low\nTopic: returns\nSuggested reply:\n- You can return the jacket within 30 days as long as it is unworn and in its original packaging.\n- Start the return from your account dashboard and we will email you a prepaid label.",
}

few_shot_prompt_v2 = FewShotPromptTemplate(
    examples=examples + [returns_example],
    example_prompt=example_prompt,
    prefix=prefix,
    suffix=suffix,
    input_variables=["message"],
    example_separator="\n\n",
)

few_shot_chain_v2 = few_shot_prompt_v2 | llm | output_parser

returns_cases = [
    {
        "label": "Late refund",
        "message": "I returned my shoes last week but I still have not received the refund.",
        "watch_for": "Returns or billing? The return is done, the money is missing.",
    },
    {
        "label": "Damaged, wants refund",
        "message": "My package was damaged in transit and I want to send it back for a refund.",
        "watch_for": "Returns or shipping? The cause is shipping, the request is a return.",
    },
    {
        "label": "Plain return",
        "message": "These headphones are not what I expected. What is the return process?",
        "watch_for": "Clear returns case, low urgency.",
    },
]
returns_inputs = [{"message": case["message"]} for case in returns_cases]

outputs_two = few_shot_chain.batch(returns_inputs, config={"max_concurrency": 2})
outputs_three = few_shot_chain_v2.batch(returns_inputs, config={"max_concurrency": 2})

print("#" * 30, "TWO EXAMPLES (original)", "#" * 30)
print_outputs(returns_cases, outputs_two)
print("\n" + "#" * 30, "THREE EXAMPLES (with returns)", "#" * 30)
print_outputs(returns_cases, outputs_three)

print(f"\n{'Case':<26}{'two examples':<22}{'three examples':<22}")
for case, two, three in zip(returns_cases, outputs_two, outputs_three):
    label_two = extract_label(two, "Urgency") + " / " + extract_label(two, "Topic")
    label_three = extract_label(three, "Urgency") + " / " + extract_label(three, "Topic")
    print(f"{case['label']:<26}{label_two:<22}{label_three:<22}")

############################## TWO EXAMPLES (original) ##############################
Case: Late refund
What to watch: Returns or billing? The return is done, the money is missing.
Urgency: medium  
Topic: returns  
Suggested reply:  
- I’m sorry you haven’t received your refund yet.  
- Please provide your order number and we’ll check the status and expedite the refund if needed.
Case: Damaged, wants refund
What to watch: Returns or shipping? The cause is shipping, the request is a return.
Urgency: medium  
Topic: returns  
Suggested reply:  
- I’m sorry to hear that your package arrived damaged.  
- Please provide your order number and we’ll guide you through the return and refund process.
Case: Plain return
What to watch: Clear returns case, low urgency.
Urgency: low
Topic: returns
Suggested reply:
- I'm sorry to hear the headphones didn't meet your expectations.
- You can initiate a return by visiting the Returns section in your account or by replying to this email with your order 

**What I observed** (run of 2026-11-13, `openai/gpt-oss-20b`; a rerun will differ in detail)

- **The labels did not change:** all three messages were `returns` with two examples and stayed `returns` with three, at the same urgency. There was no classification problem here for the new example to fix.
- **What changed is the reply text.** With the returns example, the answers picked up its wording: "account dashboard", "within 30 days", "prepaid return label". Those are policy details from my example, now stated to customers as fact.
- **Conclusion:** check on the test set whether an example is needed before adding it. Here it added prompt length and leaked its content into the replies without improving a single label.

### Exercise 4 - Raise the Temperature

One run per temperature proves nothing, because a single output can look fine by chance. So the same message goes through the same improved prompt three times at temperature `0.1` and three times at `1.0`. Consistency is then counted: how many different labels and how many different full texts came back from three identical requests?

In [20]:
RUNS = 3
temperature_message = support_cases[2]["message"]  # the double-charge case

temperature_results = {}
for temperature in (0.1, 1.0):
    test_llm = create_chat_model(temperature=temperature, max_tokens=512)
    test_chain = improved_prompt | test_llm | output_parser
    temperature_results[temperature] = [
        test_chain.invoke({"message": temperature_message}) for _ in range(RUNS)
    ]

for temperature, outputs in temperature_results.items():
    print("\n" + "#" * 30, f"temperature = {temperature}", "#" * 30)
    for run, output in enumerate(outputs, start=1):
        print(f"--- run {run} ---")
        print(output)

print(f"\n{'temperature':<14}{'different labels':<20}{'different full texts':<22}")
for temperature, outputs in temperature_results.items():
    labels = {(extract_label(o, "Urgency"), extract_label(o, "Topic")) for o in outputs}
    texts = {o.strip() for o in outputs}
    print(f"{temperature:<14}{len(labels):<20}{len(texts):<22}")


############################## temperature = 0.1 ##############################
--- run 1 ---
Urgency: medium  
Topic: billing  
Suggested reply:  
- I'm sorry to hear about the duplicate charge; I'll look into your order right away.  
- Once I confirm the issue, I'll issue a refund for the extra amount and let you know the status.
--- run 2 ---
Urgency: high  
Topic: billing  
Suggested reply:  
- I'm sorry to hear about the duplicate charge; let me look into this for you right away.  
- Could you please provide your order number and the date of the transaction so we can verify and resolve the issue?
--- run 3 ---
Urgency: medium  
Topic: billing  
Suggested reply:  
- I'm sorry to hear about the duplicate charge; let me look into it right away.  
- Please provide your order number and the date of the transaction, and I'll verify the charges and issue a refund if needed.

############################## temperature = 1.0 ##############################
--- run 1 ---
Urgency: high  
Top

**What I observed** (run of 2026-11-13, `openai/gpt-oss-20b`; a rerun will differ in detail)

- **The result is the opposite of what I expected.** At temperature 0.1 the three runs gave two different urgency labels (medium, high, medium); at 1.0 all three said high.
- **The full text was different in every one of the six runs,** at both temperatures. Temperature 0.1 is not deterministic on this model.
- **Three runs are too few to rank the two temperatures.** What the result does show: the double-charge case sits on the border between `medium` and `high`, and the prompt never says what the levels mean. The instability comes from that missing definition, so the fix is a criterion in the prompt (for example "high = money or a deadline is at stake"), not a temperature setting.
- **For prompt testing I would still keep the temperature low,** so that differences between variants come from the prompt as far as possible, and repeat each case several times instead of trusting one run.

## Conclusion

In this notebook, you practiced a simple but powerful prompt-engineering workflow:
- define a task clearly
- create a compact evaluation set
- compare prompt variants on the same cases
- tighten the prompt format when outputs are inconsistent
- add few-shot examples when you need even more control

This kind of lightweight testing loop is a strong next step after learning the LangChain basics and the core prompt-engineering patterns.